In [ ]:
%pip install arxiv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6090 sha256=410c9ca59f67d6c95fecfaa030ae74d818209742e83126411c14c6b39b50fae2
  Stored in directory: /Users/nithingedda/Library/Caches/pip/wheels/e3/43/83/0f6e317d0698ac38ee6a5b6e214019c167057916a11bad91ab
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [arxiv]


In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


In [2]:
# Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# Retriever Tool
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, OllamaEmbeddings(model="llama3:latest"))
retriever = vectordb.as_retriever()


In [5]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever, "Langsmith_search","Search for information about Langsmith.")

In [6]:
retriever_tool.name

'Langsmith_search'

In [7]:
# Arxiv tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

axriv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
axriv_tool = ArxivQueryRun(axriv_wrapper=axriv_wrapper)
axriv_tool.name


'arxiv'

In [8]:
tools = [wiki_tool,retriever_tool,axriv_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/nithingedda/Desktop/LangChain/myenv/lib/python3.14/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='Langsmith_search', description='Search for information about Langsmith.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x10b05ab90>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x113b5da60>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000))]

In [29]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

In [30]:
#from langchain_core.prompts import ChatPromptTemplate
#from langchain_community.utilities import hub
#from langchain_community import hub
#from langchain import hub

#prompt = hub.pull("hwchase17/react")
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""Answer the following questions as best you can. 
You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer.
Thought: you should always think about what to do.
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought: {agent_scratchpad}
""")


In [33]:
# Agents
from langchain.agents import create_agent

agent = create_agent(llm, tools)

response = agent.invoke({"messages":[("human","what is langsmith?")]})
response["messages"][-1].content

'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It allows users to create, test, and deploy their AI models with ease, while also providing features such as prompt engineering, trace requests, and infrastructure setup. LangSmith offers a visual interface for designing and testing applications end-to-end, as well as a CLI for querying and managing traces, datasets, experiments, and more from the terminal. The platform is designed to meet high standards of data security and privacy, making it suitable for a wide range of use cases.'